# Modelling: UCI Adult Census Income (1994)

Logistic Regression and Random Forest trained on the prepared UCI frame, each evaluated for accuracy and five fairness metrics across two protected attributes (`sex`, `race_binary`).

All runs go through one reusable function, `fit_and_evaluate` (Section 0), so any difference in results is due to the model or experiment, not the code path.

**Experiments**
- Baseline: all features included.
- Protected attributes removed (naive): `sex` and `race_binary` dropped, proxies kept.
- Protected attributes and proxies removed: `sex`, `race_binary` and their strongest proxies dropped.

Results exported in long format to `results/tables/uci_model_results.csv` for the comparison notebook (06).

## 0. Setup

Imports and the reusable `fit_and_evaluate` function.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

%pip install fairlearn

from fairlearn.metrics import (
    MetricFrame,
    selection_rate,
    true_positive_rate,
    false_positive_rate,
    demographic_parity_difference,
    equalized_odds_difference,
)

SEED = 42
DATASET = "UCI Adult (1994)"
RESULTS_DIR = "../results/tables"

pd.set_option("display.width", 120)

Note: you may need to restart the kernel to use updated packages.


In [2]:
def fit_and_evaluate(model, preprocessor, X_tr, y_tr, X_te, y_te,
                     sensitive_cols, model_name, experiment, dataset_name):
    """Fit one model, predict on the test set, and compute accuracy plus the five
    fairness metrics per protected attribute.

    Returns long-format rows {dataset, model, experiment, protected_attribute,
    metric, value}, one per (protected attribute x metric), to stack in notebook 06.

    Sensitive-feature columns are read from the raw X_te (pre-encoding), since the
    fairness grouping needs the original labels (e.g. 'Male'/'Female')
    """
    X_tr_p = preprocessor.fit_transform(X_tr)
    X_te_p = preprocessor.transform(X_te)
    model.fit(X_tr_p, y_tr)
    y_pred = model.predict(X_te_p)

    rows = []
    acc = accuracy_score(y_te, y_pred)

    for attr in sensitive_cols:
        sf = X_te[attr].to_numpy()

        # SPD and Equalised Odds: Fairlearn named functions.
        spd = demographic_parity_difference(y_te, y_pred, sensitive_features=sf)
        eqo = equalized_odds_difference(y_te, y_pred, sensitive_features=sf)

        # DI, EOD, Predictive Parity: built from grouped rates via MetricFrame.
        mf = MetricFrame(
            metrics={
                "selection_rate": selection_rate,
                "tpr": true_positive_rate,
                "precision": precision_score,
            },
            y_true=y_te, y_pred=y_pred, sensitive_features=sf,
        )
        by = mf.by_group
        di = by["selection_rate"].min() / by["selection_rate"].max()   # Disparate Impact ratio
        eod = by["tpr"].max() - by["tpr"].min()                        # Equal Opportunity Diff
        pp = by["precision"].max() - by["precision"].min()             # Predictive Parity gap

        for metric_name, value in [
            ("accuracy", acc),
            ("SPD", spd),
            ("DI", di),
            ("EOD", eod),
            ("EqualisedOdds", eqo),
            ("PredictiveParity", pp),
        ]:
            rows.append({
                "dataset": dataset_name,
                "model": model_name,
                "experiment": experiment,
                "protected_attribute": attr,
                "metric": metric_name,
                "value": round(float(value), 4),
            })

    return rows

## 1. Load and split

The prepared frame exported by notebook 01 is loaded, the target is mapped to 0/1, columns
that must not be modelled are dropped, and a stratified 75/25 split preserves the class
imbalance in both train and test.

In [3]:
df = pd.read_csv("../data/processed/uci_1994_prepared.csv")

# Target: ">50K" is the positive class (1).
y = (df["income"] == ">50K").astype(int)

# Dropped from the feature matrix
DROP = ["fnlwgt", "hours_bucket", "education", "race", "race_cluster", "income"]
X = df.drop(columns=DROP)

PROTECTED = ["sex", "race_binary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

print("train:", X_train.shape, "| test:", X_test.shape)
print("train positive rate:", round(y_train.mean(), 4),
      "| test positive rate:", round(y_test.mean(), 4))
print("feature columns:", X.columns.tolist())

train: (36631, 12) | test: (12211, 12)
train positive rate: 0.2393 | test positive rate: 0.2393
feature columns: ['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'relationship', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'race_binary']


## 2. Preprocessing

Numeric features standardised, categoricals one-hot encoded. The transformer is fitted *inside* `fit_and_evaluate` on each call , on the training columns only, so no test-set information leaks and every experiment fits its own transformer.

In [4]:
NUMERIC = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
CATEGORICAL = ["workclass", "marital-status", "occupation", "relationship",
               "sex", "native-country", "race_binary"]

assert set(NUMERIC + CATEGORICAL) == set(X_train.columns), "column role mismatch"


def make_preprocessor(numeric, categorical):
    """Build a fresh ColumnTransformer. each experiment
    gets its own unfitted preprocessor over whatever columns that experiment uses."""
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
        ],
        remainder="drop",
    )


# Quick check that the full-feature preprocessor produces matching train/test shapes.
_pp = make_preprocessor(NUMERIC, CATEGORICAL)
_tr = _pp.fit_transform(X_train)
_te = _pp.transform(X_test)
print("processed train:", _tr.shape, "| processed test:", _te.shape,
      "| equal cols:", _tr.shape[1] == _te.shape[1])

processed train: (36631, 88) | processed test: (12211, 88) | equal cols: True


In [5]:
# Confirm what the 88 columns actually are.
pp = make_preprocessor(NUMERIC, CATEGORICAL)
pp.fit(X_train)
n_num = len(NUMERIC)
n_cat = pp.named_transformers_["cat"].get_feature_names_out(CATEGORICAL).shape[0]
print(f"numeric: {n_num} + categorical one-hot: {n_cat} = {n_num + n_cat}")
for col in CATEGORICAL:
    print(f"  {col}: {X_train[col].nunique()} levels")

numeric: 5 + categorical one-hot: 83 = 88
  workclass: 9 levels
  marital-status: 7 levels
  occupation: 15 levels
  relationship: 6 levels
  sex: 2 levels
  native-country: 42 levels
  race_binary: 2 levels


## 3. Baseline

Both models trained on the full feature set (protected attributes included). The reference point for the later experiments.

In [6]:
results = []  # accumulates every experiment's rows; exported at the end

lr = LogisticRegression(max_iter=1000, random_state=SEED)
results += fit_and_evaluate(
    lr, make_preprocessor(NUMERIC, CATEGORICAL),
    X_train, y_train, X_test, y_test,
    sensitive_cols=PROTECTED,
    model_name="LogisticRegression", experiment="baseline", dataset_name=DATASET,
)

rf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
results += fit_and_evaluate(
    rf, make_preprocessor(NUMERIC, CATEGORICAL),
    X_train, y_train, X_test, y_test,
    sensitive_cols=PROTECTED,
    model_name="RandomForest", experiment="baseline", dataset_name=DATASET,
)

baseline_df = pd.DataFrame(results)
# Show as a readable grid: metrics down the side, model x attribute across the top.
baseline_pivot = baseline_df.pivot_table(
    index="metric",
    columns=["model", "protected_attribute"],
    values="value",
)
print(baseline_pivot.round(4))

model               LogisticRegression         RandomForest        
protected_attribute        race_binary     sex  race_binary     sex
metric                                                             
DI                              0.5409  0.3002       0.5462  0.3266
EOD                             0.1021  0.1063       0.0754  0.0950
EqualisedOdds                   0.1021  0.1063       0.0754  0.0950
PredictiveParity                0.0463  0.0210       0.0140  0.0221
SPD                             0.0948  0.1750       0.0983  0.1747
accuracy                        0.8543  0.8543       0.8519  0.8519


In [7]:
# Verify EOD vs EqualisedOdds are legitimately equal (TPR gap >= FPR gap)
from fairlearn.metrics import MetricFrame, true_positive_rate, false_positive_rate

pp = make_preprocessor(NUMERIC, CATEGORICAL)
Xtr = pp.fit_transform(X_train); Xte = pp.transform(X_test)
lr_check = LogisticRegression(max_iter=1000, random_state=SEED).fit(Xtr, y_train)
yp = lr_check.predict(Xte)

for attr in PROTECTED:
    sf = X_test[attr].to_numpy()
    mf = MetricFrame(metrics={"tpr": true_positive_rate, "fpr": false_positive_rate},
                     y_true=y_test, y_pred=yp, sensitive_features=sf)
    tpr_gap = mf.by_group["tpr"].max() - mf.by_group["tpr"].min()
    fpr_gap = mf.by_group["fpr"].max() - mf.by_group["fpr"].min()
    print(f"{attr}: TPR gap={tpr_gap:.4f}, FPR gap={fpr_gap:.4f}, "
          f"EqOdds should be max={max(tpr_gap, fpr_gap):.4f}")

sex: TPR gap=0.1063, FPR gap=0.0727, EqOdds should be max=0.1063
race_binary: TPR gap=0.1021, FPR gap=0.0304, EqOdds should be max=0.1021


## 4. Protected attributes removed

In [8]:
# Protected attributes removed, in two forms:
# (i) naive: drop only the protected attributes themselves.
# (ii) proxies: also drop their strongest proxies (relationship for sex, native-country
#      for race), from the UCI EDA proxy-strength ranking.

# --- naive removal ---
NUM_B1 = [c for c in NUMERIC if c not in ("sex", "race_binary")]        # numerics unaffected
CAT_B1 = [c for c in CATEGORICAL if c not in ("sex", "race_binary")]

for name, model in [("LogisticRegression", LogisticRegression(max_iter=1000, random_state=SEED)),
                    ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))]:
    results += fit_and_evaluate(
        model, make_preprocessor(NUM_B1, CAT_B1),
        X_train, y_train, X_test, y_test,
        sensitive_cols=PROTECTED,          # still AUDIT on sex/race, even though not used as features
        model_name=name, experiment="protected_removed_naive", dataset_name=DATASET,
    )

# --- removal + top proxies ---
PROXIES = ["relationship", "native-country"]
NUM_B2 = [c for c in NUMERIC if c not in (["sex", "race_binary"] + PROXIES)]
CAT_B2 = [c for c in CATEGORICAL if c not in (["sex", "race_binary"] + PROXIES)]

for name, model in [("LogisticRegression", LogisticRegression(max_iter=1000, random_state=SEED)),
                    ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))]:
    results += fit_and_evaluate(
        model, make_preprocessor(NUM_B2, CAT_B2),
        X_train, y_train, X_test, y_test,
        sensitive_cols=PROTECTED,
        model_name=name, experiment="protected_removed_proxies", dataset_name=DATASET,
    )

# Compare all three experiments on the key fairness metrics, for sex.
comp = pd.DataFrame(results)
sex_view = comp[comp["protected_attribute"] == "sex"].pivot_table(
    index="metric", columns=["experiment", "model"], values="value")
print("SEX across experiments:")
print(sex_view.round(4))

SEX across experiments:
experiment                 baseline              protected_removed_naive              protected_removed_proxies  \
model            LogisticRegression RandomForest      LogisticRegression RandomForest        LogisticRegression   
metric                                                                                                            
DI                           0.3002       0.3266                  0.3178       0.3463                    0.2665   
EOD                          0.1063       0.0950                  0.0789       0.0793                    0.1576   
EqualisedOdds                0.1063       0.0950                  0.0789       0.0793                    0.1576   
PredictiveParity             0.0210       0.0221                  0.0157       0.0426                    0.0297   
SPD                          0.1750       0.1747                  0.1675       0.1698                    0.1833   
accuracy                     0.8543       0.8519        

In [9]:
# Sanity check: inspect the raw group selection rates and TPRs behind the SPD/EOD numbers,
# across the three experiments, for LR on sex. If the "proxy removal makes it worse" story
# is real, we should SEE the female selection rate fall (or male rise) when relationship goes.
from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate

configs = {
    "baseline":        (NUMERIC, CATEGORICAL),
    "naive":           (NUM_B1, CAT_B1),
    "proxies":         (NUM_B2, CAT_B2),
}

sf_sex = X_test["sex"].to_numpy()
print(f"{'experiment':<14} {'F sel':>7} {'M sel':>7} {'SPD':>7} {'F tpr':>7} {'M tpr':>7} {'acc':>7}")
for label, (num, cat) in configs.items():
    pp = make_preprocessor(num, cat)
    Xtr = pp.fit_transform(X_train); Xte = pp.transform(X_test)
    m = LogisticRegression(max_iter=1000, random_state=SEED).fit(Xtr, y_train)
    yp = m.predict(Xte)
    mf = MetricFrame(metrics={"sel": selection_rate, "tpr": true_positive_rate},
                     y_true=y_test, y_pred=yp, sensitive_features=sf_sex)
    sel = mf.by_group["sel"]; tpr = mf.by_group["tpr"]
    acc = (yp == y_test).mean()
    print(f"{label:<14} {sel['Female']:>7.4f} {sel['Male']:>7.4f} "
          f"{sel['Male']-sel['Female']:>7.4f} {tpr['Female']:>7.4f} {tpr['Male']:>7.4f} {acc:>7.4f}")

# Also confirm relationship really was carrying signal: how many features in each config?
for label, (num, cat) in configs.items():
    pp = make_preprocessor(num, cat); pp.fit(X_train)
    n = pp.transform(X_test).shape[1]
    print(f"{label}: {n} features")

experiment       F sel   M sel     SPD   F tpr   M tpr     acc
baseline        0.0750  0.2500  0.1750  0.5077  0.6140  0.8543
naive           0.0780  0.2455  0.1675  0.5254  0.6043  0.8537
proxies         0.0666  0.2499  0.1833  0.4547  0.6124  0.8526
baseline: 88 features
naive: 84 features
proxies: 36 features


## 5. Export

In [10]:
uci_results = pd.DataFrame(results)

key = ["model", "experiment", "protected_attribute", "metric"]
dupes = uci_results.duplicated(subset=key).sum()
print("duplicate rows:", dupes, "(should be 0)")
if dupes:
    print("WARNING: results list contains duplicates; restart kernel and Run All to fix.")

print("experiments present:", sorted(uci_results["experiment"].unique()))
print("total rows:", len(uci_results))

uci_results.to_csv(f"{RESULTS_DIR}/uci_model_results.csv", index=False)
print(f"written: {RESULTS_DIR}/uci_model_results.csv")

duplicate rows: 0 (should be 0)
experiments present: ['baseline', 'protected_removed_naive', 'protected_removed_proxies']
total rows: 72
written: ../results/tables/uci_model_results.csv


## 6. Additional analyses (bootstrap CIs, intersectional)

Bootstrap 95% CIs on the baseline fairness metrics, and a race × sex intersectional audit. Each writes its own CSV and leaves the main export untouched.

In [11]:
# Bootstrap 95% CIs for the fairness metrics.
# Resample the test set with replacement, recompute each metric per resample,
# and take the 2.5th and 97.5th percentiles. The model is trained once; only
# evaluation is resampled.
from sklearn.metrics import precision_score
from fairlearn.metrics import (MetricFrame, selection_rate,
                               true_positive_rate, demographic_parity_difference,
                               equalized_odds_difference)

def bootstrap_fairness_cis(model, preprocessor, X_tr, y_tr, X_te, y_te,
                           sensitive_cols, model_name, experiment, dataset_name,
                           n_boot=1000, seed=42):
    """Train once, then bootstrap the test set n_boot times. Returns long-format rows:
    {dataset, model, experiment, protected_attribute, metric, value,
     ci_low, ci_high, n_test, n_pos}."""
    rng = np.random.default_rng(seed)

    X_tr_p = preprocessor.fit_transform(X_tr)
    X_te_p = preprocessor.transform(X_te)
    model.fit(X_tr_p, y_tr)

    y_te = np.asarray(y_te)
    idx_all = np.arange(len(y_te))

    def metrics_on(idx, y_pred_full):
        yt = y_te[idx]; yp = y_pred_full[idx]
        out = {}
        for attr in sensitive_cols:
            sf = X_te[attr].to_numpy()[idx]
            # guard: a resample might miss a group entirely
            if len(np.unique(sf)) < 2:
                out[attr] = None; continue
            spd = demographic_parity_difference(yt, yp, sensitive_features=sf)
            eqo = equalized_odds_difference(yt, yp, sensitive_features=sf)
            mf = MetricFrame(metrics={"sel": selection_rate, "tpr": true_positive_rate,
                                      "prec": lambda yt,yp: precision_score(yt,yp,zero_division=0)},
                             y_true=yt, y_pred=yp, sensitive_features=sf)
            by = mf.by_group
            di  = by["sel"].min() / by["sel"].max() if by["sel"].max() > 0 else np.nan
            eod = by["tpr"].max() - by["tpr"].min()
            pp  = by["prec"].max() - by["prec"].min()
            out[attr] = {"SPD": spd, "DI": di, "EOD": eod,
                         "EqualisedOdds": eqo, "PredictiveParity": pp}
        return out

    y_pred_full = model.predict(X_te_p)
    point = metrics_on(idx_all, y_pred_full)

    # bootstrap
    boot = {attr: {m: [] for m in ["SPD","DI","EOD","EqualisedOdds","PredictiveParity"]}
            for attr in sensitive_cols}
    for _ in range(n_boot):
        idx = rng.choice(idx_all, size=len(idx_all), replace=True)
        res = metrics_on(idx, y_pred_full)
        for attr in sensitive_cols:
            if res[attr] is None: continue
            for m, v in res[attr].items():
                if v is not None and not np.isnan(v):
                    boot[attr][m].append(v)

    rows = []
    for attr in sensitive_cols:
        sf_full = X_te[attr].to_numpy()
        n_test = len(sf_full)
        n_pos = int(y_te.sum())
        for m in ["SPD","DI","EOD","EqualisedOdds","PredictiveParity"]:
            vals = np.array(boot[attr][m])
            lo, hi = (np.percentile(vals, [2.5, 97.5]) if len(vals) else (np.nan, np.nan))
            rows.append({"dataset": dataset_name, "model": model_name,
                         "experiment": experiment, "protected_attribute": attr,
                         "metric": m, "value": round(float(point[attr][m]), 4),
                         "ci_low": round(float(lo), 4), "ci_high": round(float(hi), 4),
                         "n_test": n_test, "n_pos": n_pos})
    return rows

print("bootstrap_fairness_cis defined")


bootstrap_fairness_cis defined


In [12]:
CI_FILE = "uci_ci_baseline.csv"

# Bootstrap CIs on the baseline models

ci_rows = []
for name, model in [("LogisticRegression", LogisticRegression(max_iter=1000, random_state=SEED)),
                    ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))]:
    ci_rows += bootstrap_fairness_cis(
        model, make_preprocessor(NUMERIC, CATEGORICAL),
        X_train, y_train, X_test, y_test,
        sensitive_cols=PROTECTED, model_name=name,
        experiment="baseline", dataset_name=DATASET,
        n_boot=1000, seed=SEED)

ci_df = pd.DataFrame(ci_rows)
print("Baseline fairness metrics with 95% bootstrap CIs:")
print(ci_df[["model","protected_attribute","metric","value","ci_low","ci_high","n_test","n_pos"]].to_string(index=False))
ci_df.to_csv(f"{RESULTS_DIR}/{CI_FILE}", index=False)
print(f"\nwritten: {RESULTS_DIR}/{CI_FILE}")


Baseline fairness metrics with 95% bootstrap CIs:
             model protected_attribute           metric  value  ci_low  ci_high  n_test  n_pos
LogisticRegression                 sex              SPD 0.1750  0.1639   0.1861   12211   2922
LogisticRegression                 sex               DI 0.3002  0.2678   0.3313   12211   2922
LogisticRegression                 sex              EOD 0.1063  0.0560   0.1558   12211   2922
LogisticRegression                 sex    EqualisedOdds 0.1063  0.0719   0.1558   12211   2922
LogisticRegression                 sex PredictiveParity 0.0210  0.0014   0.0724   12211   2922
LogisticRegression         race_binary              SPD 0.0948  0.0785   0.1101   12211   2922
LogisticRegression         race_binary               DI 0.5409  0.4732   0.6154   12211   2922
LogisticRegression         race_binary              EOD 0.1021  0.0403   0.1598   12211   2922
LogisticRegression         race_binary    EqualisedOdds 0.1021  0.0407   0.1598   12211   2922


In [13]:
INTER_FILE = "uci_intersectional.csv"
# Intersectional analysis: race x sex on the baseline models.
# A model that is fair on race and on sex separately can still be unfair to a
# specific combination (e.g. Black women). A combined race_binary x sex group is
# built and the baseline models are audited on it.
from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate
from sklearn.metrics import precision_score

def intersectional_eval(model, preprocessor, X_tr, y_tr, X_te, y_te,
                        model_name, dataset_name):
    """Audit one trained baseline model across race_binary × sex subgroups.
    Returns per-subgroup rows plus overall max-gap rows."""
    X_tr_p = preprocessor.fit_transform(X_tr)
    X_te_p = preprocessor.transform(X_te)
    model.fit(X_tr_p, y_tr)
    y_pred = model.predict(X_te_p)
    yt = np.asarray(y_te)

    inter = (X_te["race_binary"].astype(str) + " / " + X_te["sex"].astype(str)).to_numpy()
    mf = MetricFrame(metrics={"selection_rate": selection_rate,
                              "tpr": true_positive_rate,
                              "precision": lambda yt,yp: precision_score(yt,yp,zero_division=0)},
                     y_true=yt, y_pred=y_pred, sensitive_features=inter)
    by = mf.by_group

    rows = []
    for grp in by.index:
        n = int((inter == grp).sum())
        rows.append({"dataset": dataset_name, "model": model_name,
                     "experiment": "baseline_intersectional", "subgroup": grp,
                     "n": n,
                     "selection_rate": round(float(by.loc[grp, "selection_rate"]), 4),
                     "tpr": round(float(by.loc[grp, "tpr"]), 4),
                     "precision": round(float(by.loc[grp, "precision"]), 4)})
    # overall gaps across the four subgroups
    rows.append({"dataset": dataset_name, "model": model_name,
                 "experiment": "baseline_intersectional", "subgroup": "GAP (max-min)",
                 "n": len(inter),
                 "selection_rate": round(float(by["selection_rate"].max() - by["selection_rate"].min()), 4),
                 "tpr": round(float(by["tpr"].max() - by["tpr"].min()), 4),
                 "precision": round(float(by["precision"].max() - by["precision"].min()), 4)})
    return rows

inter_rows = []
for name, model in [("LogisticRegression", LogisticRegression(max_iter=1000, random_state=SEED)),
                    ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))]:
    inter_rows += intersectional_eval(
        model, make_preprocessor(NUMERIC, CATEGORICAL),
        X_train, y_train, X_test, y_test, name, DATASET)

inter_df = pd.DataFrame(inter_rows)
print("Intersectional (race_binary x sex) baseline:")
print(inter_df.to_string(index=False))
inter_df.to_csv(f"{RESULTS_DIR}/{INTER_FILE}", index=False)
print(f"\nwritten: {RESULTS_DIR}/{INTER_FILE}")


Intersectional (race_binary x sex) baseline:
         dataset              model              experiment           subgroup     n  selection_rate    tpr  precision
UCI Adult (1994) LogisticRegression baseline_intersectional Non-White / Female   804          0.0448 0.4483     0.7222
UCI Adult (1994) LogisticRegression baseline_intersectional   Non-White / Male  1022          0.1644 0.5200     0.6964
UCI Adult (1994) LogisticRegression baseline_intersectional     White / Female  3220          0.0826 0.5165     0.7669
UCI Adult (1994) LogisticRegression baseline_intersectional       White / Male  7165          0.2622 0.6234     0.7445
UCI Adult (1994) LogisticRegression baseline_intersectional      GAP (max-min) 12211          0.2175 0.1752     0.0705
UCI Adult (1994)       RandomForest baseline_intersectional Non-White / Female   804          0.0398 0.4138     0.7500
UCI Adult (1994)       RandomForest baseline_intersectional   Non-White / Male  1022          0.1800 0.5778     0.7065
UCI